### Coalition Truncation at Scale — Sobol' G-Function ($d=10$)

The Sobol' G-function is a standard benchmark for high-dimensional
sensitivity analysis.  With tunable importance weights $a_i$, it
creates a clear hierarchy of variable importance:

$$
f(\mathbf{x}) = \prod_{i=1}^{d} \frac{|4x_i - 2| + a_i}{1 + a_i},
\quad x_i \in [0, 1]
$$

**Why $d=10$?**  The full exhaustive method enumerates $2^{10} - 1 =
1{,}023$ subsets.  At $N=3000$ samples each, that's ~6 million model
evaluations.  With $k_{\max}=2$, only $\binom{10}{1} + \binom{10}{2} + 1
= 56$ subsets are evaluated — an **18× reduction**.

This notebook demonstrates:
1. Training an RS-HDMR surrogate on $d=10$ G-function data
2. Auto-detected $k_{\max}=3$ (from `polys=[8, 5, 3]`) vs full 1,023
3. Explicit $k_{\max}=2$ as a controlled approximation
4. Validation against analytical Sobol' indices

In [ ]:
# Import dependencies
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from scipy.stats import qmc

from shapleyx import rshdmr
from shapleyx.utilities.mc_shapley import coalitions_up_to_k

from importlib.metadata import version
print(f"ShapleyX v{version('shapleyx')}")

---
### Sobol' G-Function Setup ($d=10$)

Weights $a_i$ are chosen to create a steep importance hierarchy:
$a = [0, 0.5, 1, 2, 3, 4, 6, 8, 10, 12]$.  Smaller $a_i$ → more
influential variable.  Analytical Sobol' indices are known in
closed form [1], providing ground truth for validation.

---
[1] Saltelli, A. et al. (2004). *Sensitivity Analysis in Practice.*
Wiley, Ch. 2.

In [ ]:
d = 10
a = np.array([0.0, 0.5, 1.0, 2.0, 3.0, 4.0, 6.0, 8.0, 10.0, 12.0])

def g_function(X):
    """Sobol' G-function — vectorised over rows."""
    return np.prod((np.abs(4 * X - 2) + a) / (1 + a), axis=1)

def g_function_1d(x):
    """Single-sample wrapper for MC Shapley."""
    return float(np.prod((np.abs(4 * np.asarray(x) - 2) + a) / (1 + a)))

# Analytical first-order Sobol' indices
V_i = 1 / (3 * (1 + a)**2)
total_var = np.prod(1 + V_i) - 1
S_analytical = V_i / total_var * np.prod(1 + V_i) / (1 + V_i)
# Simplified: S_i = V_i / sum(V_j) for independent product form
# Actually: S_i = V_i / (∏(1+V_j) - 1) * ∏_{j≠i}(1+V_j)
prod_all = np.prod(1 + V_i)
S_analytical = np.array([
    V_i[i] * prod_all / (1 + V_i[i]) / (prod_all - 1)
    for i in range(d)
])

print("Analytical first-order Sobol' indices:")
for i in range(d):
    print(f"  X{i+1}: {S_analytical[i]:.4f}")

In [ ]:
# Generate training data — 1024 Sobol' samples
sampler = qmc.Sobol(d, scramble=True, seed=123)
X_train = sampler.random_base2(10)  # 2^10 = 1024 samples
Y_train = g_function(X_train)

cols = [f'X{i+1}' for i in range(d)]
df = pd.DataFrame(X_train, columns=cols)
df['Y'] = Y_train
print(f"{len(df)} training samples")

In [ ]:
# Train RS-HDMR surrogate — polys=[8,5,3] has up to 3rd-order interactions
model = rshdmr(
    df,
    polys=[8, 5, 3],
    n_iter=80,
    method='ard_cv',
    cv_tol=0.005,
)
sob, shap, total = model.run_all()
print(f"\nInteraction order from polys: {len(model.polys)} → k_max={len(model.polys)}")

---
### Subset Reduction at $d=10$

The benefit of coalition truncation grows dramatically with dimension.

In [ ]:
print(f"Full enumeration:          {len(coalitions_up_to_k(d, None)):>5,d} subsets")
for kmax in [1, 2, 3, 4]:
    n = len(coalitions_up_to_k(d, kmax))
    pct = 100 * n / len(coalitions_up_to_k(d, None))
    speedup = len(coalitions_up_to_k(d, None)) / n
    print(f"k_max={kmax}: {n:>5,d} subsets ({pct:5.1f}% of full, {speedup:.0f}× reduction)")

---
### MC Shapley: Full vs Auto-Detected $k_{\max}=3$

The surrogate has up to 3rd-order interactions.  We compare full
enumeration against the auto-detected truncation and an aggressive
$k_{\max}=2$ approximation.

In [ ]:
N_mc = 2000  # smaller N per subset to keep runtime manageable

print("=" * 60)
print("Full enumeration (1,023 subsets)")
print("=" * 60)
t0 = time.time()
mc_full = model.get_mc_shapley(N=N_mc, method='exhaustive',
                              k_max=None, random_state=42)
t_full = time.time() - t0

print(f"\nTime: {t_full:.0f}s")
print(f"Subsets: 1,023")

In [ ]:
print("=" * 60)
print("Auto-detected k_max=3 (176 subsets)")
print("=" * 60)
t0 = time.time()
mc_k3 = model.get_mc_shapley(N=N_mc, method='exhaustive',
                             random_state=42)  # auto-detects k_max=3
t_k3 = time.time() - t0

print(f"\nTime: {t_k3:.0f}s (speedup: {t_full/t_k3:.1f}×)")
print(f"Subsets: 176 (vs 1,023)")

In [ ]:
print("=" * 60)
print("Explicit k_max=2 (56 subsets, approx)")
print("=" * 60)
t0 = time.time()
mc_k2 = model.get_mc_shapley(N=N_mc, method='exhaustive',
                             k_max=2, random_state=42)
t_k2 = time.time() - t0

print(f"\nTime: {t_k2:.0f}s (speedup: {t_full/t_k2:.1f}×)")
print(f"Subsets: 56 (vs 1,023)")

---
### Results Comparison

In [ ]:
comparison = pd.DataFrame({
    'Variable': mc_full['variable'],
    'Analytical S_i': S_analytical.round(4),
    'Full (1,023)': mc_full['effect'].round(4),
    'k_max=3 (176)': mc_k3['effect'].round(4),
    'k_max=2 (56)': mc_k2['effect'].round(4),
})
comparison['Diff (Full vs k3)'] = np.abs(
    mc_full['effect'] - mc_k3['effect']).round(4)
comparison['Diff (Full vs k2)'] = np.abs(
    mc_full['effect'] - mc_k2['effect']).round(4)
comparison

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(d)
w = 0.2

ax.bar(x - 1.5*w, S_analytical, w,
       color='lightgray', alpha=0.7, label='Analytical $S_i$')
ax.bar(x - 0.5*w, mc_full['effect'], w,
       color='#2166ac', alpha=0.85, label='Full (1,023 subsets)')
ax.bar(x + 0.5*w, mc_k3['effect'], w,
       color='#4393c3', alpha=0.85, label='k_max=3 (176 subsets)')
ax.bar(x + 1.5*w, mc_k2['effect'], w,
       color='#92c5de', alpha=0.85, label='k_max=2 (56 subsets)')

ax.set_xlabel('Input Variable')
ax.set_ylabel('Shapley Effect')
ax.set_title('Sobol G-Function (d=10): Coalition Truncation\n'
             f'Full={t_full:.0f}s | k_max=3={t_k3:.0f}s ({t_full/t_k3:.1f}×) | k_max=2={t_k2:.0f}s ({t_full/t_k2:.1f}×)')
ax.set_xticks(x)
ax.set_xticklabels(mc_full['variable'])
ax.legend(fontsize=8, ncol=2)
ax.set_ylim(0, None)
plt.tight_layout()
plt.show()

---
### Key Takeaways

1. **At $d=10$, $k_{\max}=2$ evaluates 56 subsets instead of 1,023** —
   an 18× reduction with results close to the full enumeration.
2. **Auto-detected $k_{\max}=3$ (from `polys=[8,5,3]`)** evaluates 176
   subsets — a 6× reduction that is exact for this surrogate.
3. **Shapley effects preserve the importance ranking** even under
   aggressive truncation ($k_{\max}=2$), though the absolute values
   shift slightly as high-order interactions are redistributed.
4. **The RS-HDMR surrogate faithfully captures the G-function's
   sensitivity structure** — the Shapley ranking mirrors the
   analytical Sobol' hierarchy ($X_1 > X_2 > X_3 > \ldots$).
5. **Coalition truncation makes exhaustive MC Shapley feasible at
   moderate-to-high dimensions** — without it, $d=10$ with
   $N=2000$ requires ~6 million evals; with $k_{\max}=2$, ~340K.

In [ ]:
%load_ext watermark
%watermark -n -u -v -iv -w